In [6]:
from pathlib import Path
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

# Use the merged model folder from outputs.
# Change this folder name to your real merged model folder.
MERGED_MODEL_DIR = Path("./outputs/qwen35-floorplan-lora/merged-best")

# Use the merged folder for processor if tokenizer/processor files were saved there.
# If that fails later, change this to Path("models/Qwen3.5-9B")
PROCESSOR_DIR = MERGED_MODEL_DIR

LOAD_IN_4BIT = True
USE_FAST_PROCESSOR = False

# Pick the best compute dtype for the current machine.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
elif torch.cuda.is_available():
    compute_dtype = torch.float16
else:
    compute_dtype = torch.float32

model_kwargs = {
    "torch_dtype": compute_dtype if torch.cuda.is_available() else torch.float32,
    "device_map": "auto" if torch.cuda.is_available() else None,
}

# Optional 4-bit loading to reduce GPU memory use.
if LOAD_IN_4BIT and torch.cuda.is_available():
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

# Load processor.
processor = AutoProcessor.from_pretrained(
    PROCESSOR_DIR,
    use_fast=USE_FAST_PROCESSOR,
)

if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"

# Load merged model directly.
model = AutoModelForImageTextToText.from_pretrained(
    MERGED_MODEL_DIR,
    **model_kwargs,
)
model.eval()

print("Loaded merged model from:", MERGED_MODEL_DIR)
print("Loaded processor from:", PROCESSOR_DIR)

The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100%|██████████| 760/760 [00:06<00:00, 123.15it/s]


Loaded merged model from: outputs/qwen35-floorplan-lora/merged-best
Loaded processor from: outputs/qwen35-floorplan-lora/merged-best


In [10]:
import io
import json
import re
from pathlib import Path
from typing import Any

from PIL import Image


MAX_IMAGE_SIDE = 1024


def resize_for_inference(image: Image.Image, max_side: int = MAX_IMAGE_SIDE) -> Image.Image:
    # Keep inference image size close to training so vision behavior stays stable.
    width, height = image.size
    longest_side = max(width, height)

    if longest_side <= max_side:
        return image

    scale = max_side / float(longest_side)
    new_size = (max(1, int(width * scale)), max(1, int(height * scale)))
    return image.resize(new_size, Image.Resampling.BICUBIC)


def clean_model_output(text: Any) -> str:
    # Normalize model output into a clean string before parsing.
    if isinstance(text, (dict, list)):
        return json.dumps(text, ensure_ascii=False)
    if text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)

    text = text.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)

    if "</think>" in text.lower():
        parts = re.split(r"</think>", text, flags=re.IGNORECASE)
        text = parts[-1].strip()

    return text.strip()


def safe_parse_json(text: Any) -> Any:
    # If already parsed, return as-is.
    if isinstance(text, (dict, list)):
        return text
    if text is None:
        return None

    text = clean_model_output(text)

    try:
        return json.loads(text)
    except Exception:
        pass

    # Try extracting the first JSON object from mixed text.
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    return None


def build_messages(question_text: str, system_prompt: str = "") -> list[dict[str, Any]]:
    # Keep message format close to training: user image + text, no hidden extra turns.
    messages = []

    if system_prompt.strip():
        messages.append(
            {
                "role": "system",
                "content": [{"type": "text", "text": system_prompt.strip()}],
            }
        )

    messages.append(
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question_text.strip()},
            ],
        }
    )
    return messages


def build_prompt(question: str, mode: str = "count", object_class: str | None = None) -> tuple[str, str]:
    # This is the main reasoning-control point for notebook inference.
    strict_system_prompt = (
        "Return only valid JSON. "
        "Do not include reasoning. "
        "Do not output <think> tags. "
        "Do not output analysis, steps, or markdown."
    )

    q = question.strip()

    if mode == "polygon":
        guided_question = (
            f"{q}\n\n"
            "Return only valid JSON using this exact schema:\n"
            '{"class":"<ObjectClass>","count":0,"present":false,"polygons":[]}'
        )
        return guided_question, strict_system_prompt

    if mode == "count":
        guided_question = (
            f"{q}\n\n"
            "Return only valid JSON using this exact schema:\n"
            '{"class":"<ObjectClass>","count":0,"present":false}'
        )
        return guided_question, strict_system_prompt

    guided_question = (
        f"{q}\n\n"
        'Return only valid JSON using this exact schema:\n{"answer":"..."}'
    )
    return guided_question, strict_system_prompt


def run_vlm(
    image_path: str | Path,
    question: str,
    mode: str = "count",
    max_new_tokens: int = 256,
    do_sample: bool = False,
    temperature: float = 0.5,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
):
    # Load and resize image.
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    image = resize_for_inference(image)

    # Build prompt with strict JSON control.
    guided_question, system_prompt = build_prompt(question=question, mode=mode)

    messages = build_messages(
        question_text=guided_question,
        system_prompt=system_prompt,
    )

    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = processor(
        text=[prompt_text],
        images=[image],
        padding=True,
        return_tensors="pt",
    )

    model_device = next(model.parameters()).device
    model_inputs = {
        key: value.to(model_device) if hasattr(value, "to") else value
        for key, value in model_inputs.items()
    }

    eos_token_id = getattr(processor.tokenizer, "eos_token_id", None) if hasattr(processor, "tokenizer") else None
    pad_token_id = getattr(processor.tokenizer, "pad_token_id", None) if hasattr(processor, "tokenizer") else None
    if pad_token_id is None and eos_token_id is not None:
        pad_token_id = eos_token_id

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
        "use_cache": True,
        "eos_token_id": eos_token_id,
        "pad_token_id": pad_token_id,
    }

    if do_sample:
        generate_kwargs["temperature"] = temperature
        generate_kwargs["top_p"] = top_p

    with torch.inference_mode():
        generated_ids = model.generate(**model_inputs, **generate_kwargs)

    prompt_len = model_inputs["input_ids"].shape[1]
    new_ids = generated_ids[:, prompt_len:]
    raw_output = processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()

    cleaned_output = clean_model_output(raw_output)
    parsed_output = safe_parse_json(cleaned_output)

    return {
        "question": question,
        "mode": mode,
        "raw_output": raw_output,
        "cleaned_output": cleaned_output,
        "parsed_output": parsed_output,
    }


In [11]:
result = run_vlm(
    image_path="raw_data/images/10064.png",
    question="how many bedrooms are there?",
    mode="count",
)

print("RAW:\n", result["raw_output"])
print("\nPARSED:\n", json.dumps(result["parsed_output"], indent=2, ensure_ascii=False))



RAW:
 The user wants to know the number of bedrooms in the provided floor plan.

1.  **Analyze the image:** I see a room with a sink, a gas stove, and a door leading into it. There is also a bathroom visible in the upper part (commode, basin, sink). The main large room looks like a living kitchen or utility area.
2.  **Identify bedroom features:** A bedroom typically has doors, windows, and no fixtures like sinks, stoves, or bathrooms.
3.  **Scan for bedrooms:**
    *   The large room at the bottom is an open space, likely a living kitchen given the sink and gas. It doesn't look like a bedroom.
    *   The room at the top contains a commode, basin, and sink. This is clearly a bathroom.
    *   There are no other rooms visible that could be classified as bedrooms.
4.  **Count:** There are 0 bedrooms.
5.  **Format output:** Return the JSON response.
</think>

{"class":"Bedroom","count":0,"present":false}

PARSED:
 {
  "class": "Bedroom",
  "count": 0,
  "present": false
}


In [9]:
result = run_vlm(
    image_path="raw_data/images/10064.png",
    question="show me all bedrooms in this floor plan and return their polygons",
    mode="polygon",
)

print("RAW:\n", result["raw_output"])
print("\nPARSED:\n", json.dumps(result["parsed_output"], indent=2, ensure_ascii=False))


RAW:
 The user wants to find all bedrooms in the provided floor plan image.

1.  **Analyze the image:**
    *   I see a room with walls, doors, windows, and fixtures.
    *   There is a bathroom (commode, basin, sink).
    *   There is a living area (large open space).
    *   There are utility areas (gas, sink).
    *   There are service elements (stairs, elevator).
    *   I need to check for "bedroom".

2.  **Scan for bedroom features:**
    *   A bedroom typically has a door leading into it, windows, and no bathroom fixtures.
    *   Looking at the layout:
        *   Top left: Wall, window.
        *   Top middle: Bathroom (basin, commode, sink visible). This is a bathroom, not a bedroom.
        *   Right side: Wall, door leading into the large room.
        *   Bottom: Living kitchen area (sink, gas), balcony (window).
        *   The large room seems to be a living kitchen or hallway, but given the door from the bathroom and the other door, it's likely a living

PARSED:
 null


In [6]:
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Base model cleared from memory.")


Base model cleared from memory.


In [7]:
print("Loading fine-tuned model...")
ft_base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=dtype,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(ft_base_model, ADAPTER_DIR)
ft_model.eval()

ft_output = generate_output(image, question, ft_model, processor)
print("Fine-tuned model done.")
print(ft_output)


Loading fine-tuned model...


Loading weights: 100%|██████████| 760/760 [00:03<00:00, 250.81it/s]


Fine-tuned model done.
Based on the visual analysis of the floor plan:

1.  **Identify the class:** The user is asking for "Junction type I", which corresponds to L-junctions.
2.  **Scan the image:** I will look for corners where two walls meet at a 90-degree angle.
3.  **Count and record polygons:**
    *   **Living Kitchen (Left side):**
        *   Top-left corner: [100.0, 336.0], [144.0, 336.0], [144.0, 360.0], [100.0, 360.0]
        *   Top-right corner (near stairs): [335.0, 336.0], [359.0, 336.0], [359.0, 360.0], [335.0, 360.0]
        *   Top-right corner (near stairs - other side): [335.0, 459.0], [359.0, 459.0], [359.0, 483.0], [335.0, 483.0]
        *   Top-right corner (near stairs - other side


In [8]:
compare_generations(base_output, ft_output)

,Base Model Output,Fine-Tuned Model Output
0,"The user wants me to analyze ""Junction type I"" in the provided floor plan. 1. **Identify ""Junction type I"":** In architectural floor plans, junctions are typically where walls meet. * **Type I (T-junction):** A wall ends at another wall (perpendicular or parallel). This is the most common type. It looks like a 'T'. * **Type II (L-junction):** Two walls meet at a 90-degree angle. * **Type III (Cross-junction):** Two walls cross each other (though less common in simple wall layouts, usually implies a specific structural element or just a cross). * **Type IV (Corner/End):** A wall ends at the end of another wall (perpendicular). Wait, let's look at standard architectural notation for junctions. * **Junction Type I:** Usually refers to a T-junction where a wall terminates into another wall. * **Junction Type II:** Usually refers to an L-junction (corner). * **Junction Type III:** Usually refers to a cross-junction (X). * **Junction Type IV:** Usually refers to a corner where a wall ends at the end of another wall (perpendicular). Let's re-evaluate based on the visual evidence. * Looking at the walls in the","Based on the visual analysis of the floor plan: 1. **Identify the class:** The user is asking for ""Junction type I"", which corresponds to L-junctions. 2. **Scan the image:** I will look for corners where two walls meet at a 90-degree angle. 3. **Count and record polygons:** * **Living Kitchen (Left side):** * Top-left corner: [100.0, 336.0], [144.0, 336.0], [144.0, 360.0], [100.0, 360.0] * Top-right corner (near stairs): [335.0, 336.0], [359.0, 336.0], [359.0, 360.0], [335.0, 360.0] * Top-right corner (near stairs - other side): [335.0, 459.0], [359.0, 459.0], [359.0, 483.0], [335.0, 483.0] * Top-right corner (near stairs - other side"
